# MLP all tasks — robustness evaluation 

This notebook runs a robustness test with the settings taken from the final MLP fine-tuning dictionary.

Workflow:

1. Run `mlp_hyperparameter_tuning.ipynb`.
2. Copy the printed `FINAL_TUNED_CFGS` dictionary.
3. Paste it into the config cell below.
4. Run this robustness notebook.

This keeps the robustness evaluation self-contained and avoids silently depending on external CSV logs.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root.parent != project_root:
    project_root = project_root.parent

if (project_root / "pyproject.toml").exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [3]:
import numpy as np
import pandas as pd
import torch

from src.data_prep import prepare_uji_data, prepare_uji_data_downgradation as prepare_uji_data_for_robustness
from src.degradation_experiments import (
    COMPOSITE_SEED_BASE,
    COMPOSITE_TRAIN_SCENARIOS,
    EVAL_DEGRADATION_SCENARIOS,
    EVAL_SEED_BASE,
    eval_scenario_seed,
)
from src.models import CoordinateMLPModel, JointMLPModel, MultiTaskMLPModel
from src.training import TrainConfig, evaluate_on_tensors, train_from_tensors


In [4]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = bundle.X_train.shape[1]

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

# Composite training data: clean + controlled perturbed copies
X_train_parts = [bundle.X_train]
for k, spec in enumerate(COMPOSITE_TRAIN_SCENARIOS):
    deg = prepare_uji_data_for_robustness(
        bundle, seed=COMPOSITE_SEED_BASE + k, **spec
    )
    X_train_parts.append(deg.X_train)

X_train_composite = np.concatenate(X_train_parts, axis=0)
n_parts = len(X_train_parts)
joint_y_composite = np.concatenate([joint_y_train] * n_parts, axis=0)
mt_y_composite = np.concatenate([mt_y_train] * n_parts, axis=0)
coord_y_composite = np.concatenate([coord_y_train] * n_parts, axis=0)

print("device:", device)
print("robustness scenarios:", len(EVAL_DEGRADATION_SCENARIOS))
print("X_train clean / composite:", bundle.X_train.shape[0], X_train_composite.shape[0])


device: cuda
robustness scenarios: 18
X_train clean / composite: 19937 79748


## Paste final fine-tuned settings

Copy `FINAL_TUNED_CFGS` from `fair_mlp_hyperparameter_tuning.ipynb` and paste it below.

Expected shape:

```python
FINAL_TUNED_CFGS = {
    "mlp": {
        "joint": {"lr": ..., "weight_decay": ..., "max_epochs": ..., "patience": ..., "batch_size": ..., "val_batch_size": ..., ...},
        "multitask": {...},
        "coordinate": {...},
    }
}
```


In [5]:
MANUAL_CONFIG_FAMILIES = ("mlp",)
TASKS = ("joint", "multitask", "coordinate")
TRAIN_KEYS = (
    "lr",
    "weight_decay",
    "max_epochs",
    "patience",
    "print_every",
    "batch_size",
    "val_batch_size",
    "grad_clip_norm",
)

# Paste the FINAL_TUNED_CFGS dictionary printed by fair_transformer_hyperparameter.ipynb here.
#
# Expected shape from the fine-tuning notebook:
# FINAL_TUNED_CFGS = {
#     "standard": {
#         "joint": {
#             "lr": ..., "weight_decay": ..., "dropout": ...,
#             "grad_clip_norm": ..., "max_epochs": ..., "patience": ...,
#             "print_every": 5, "batch_size": 256, "val_batch_size": 512,
#             "architecture": {"d_model": 128, "nhead": 4, "num_layers": 2, "dim_feedforward": 256},
#         },
#         "multitask": {...},
#         "coordinate": {...},
#     },
#     "set": {
#         "joint": {"architecture": {"d_model": 128, "num_heads": 4, "num_sab_layers": 2, "dim_feedforward": 256, "num_seed_vectors": 1}, ...},
#         ...
#     }
# }

FINAL_TUNED_CFGS = {
    'mlp': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}}
}

def validate_manual_training_configs(
    final_cfgs: dict,
    families: tuple[str, ...],
    tasks: tuple[str, ...] = TASKS,
) -> pd.DataFrame:
    if not final_cfgs:
        raise RuntimeError(
            "FINAL_TUNED_CFGS is empty. Copy the final dictionary from "
            "fair_mlp_hyperparameter_tuning_v3_original_names.ipynb first."
        )

    missing = []
    rows = []
    for family in families:
        if family not in final_cfgs:
            missing.append((family, "<family missing>"))
            continue
        for task in tasks:
            if task not in final_cfgs[family]:
                missing.append((family, task))
                continue
            spec = dict(final_cfgs[family][task])
            required = ("lr", "weight_decay", "max_epochs", "patience")
            missing_keys = [k for k in required if k not in spec]
            if missing_keys:
                raise RuntimeError(f"Config for {(family, task)} is missing keys: {missing_keys}")

            rows.append({"family": family, "task": task, **{k: spec.get(k) for k in TRAIN_KEYS}})

    if missing:
        raise RuntimeError(f"Missing required tuned configs: {missing}")

    return pd.DataFrame(rows).sort_values(["family", "task"]).reset_index(drop=True)

selected_cfg_df = validate_manual_training_configs(FINAL_TUNED_CFGS, MANUAL_CONFIG_FAMILIES)
selected_cfg_df


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm
0,mlp,coordinate,0.0005,0.0005,80,15,5,256,512,None
1,mlp,joint,0.0010,0.0005,50,10,5,256,512,None
2,mlp,multitask,0.0010,0.0005,50,10,5,256,512,None


In [6]:
def make_cfg(spec: dict, run_name: str) -> TrainConfig:
    return TrainConfig(
        run_name=run_name,
        lr=spec["lr"],
        weight_decay=spec["weight_decay"],
        batch_size=spec.get("batch_size", 256),
        val_batch_size=spec.get("val_batch_size", 512),
        max_epochs=spec["max_epochs"],
        patience=spec["patience"],
        print_every=spec.get("print_every", 5),
        grad_clip_norm=spec.get("grad_clip_norm", None),
    )

def eval_robustness_grid(
    model: torch.nn.Module,
    y_val: np.ndarray,
    eval_batch_size: int,
) -> pd.DataFrame:
    rows: list[dict] = []
    for i, (name, dr, bd, ns) in enumerate(EVAL_DEGRADATION_SCENARIOS):
        seed = eval_scenario_seed(i)
        if name == "clean":
            b = bundle
        else:
            b = prepare_uji_data_for_robustness(
                bundle,
                dropout_rate=dr,
                bias_db=bd,
                noise_std=ns,
                seed=seed,
            )
        m = evaluate_on_tensors(
            model, b.X_val, y_val, device, batch_size=eval_batch_size
        )
        rows.append({"scenario": name, **dict(m)})
    return pd.DataFrame(rows)

def stack_model_results(
    models: dict[str, torch.nn.Module],
    y_vals: dict[str, np.ndarray],
    eval_batch_sizes: dict[str, int],
    train_regime: str,
) -> pd.DataFrame:
    parts = []
    for mname, model in models.items():
        df = eval_robustness_grid(model, y_vals[mname], eval_batch_sizes[mname])
        df["model"] = mname
        df["train_regime"] = train_regime
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


## 1) Train on clean data


In [7]:
joint_clean = JointMLPModel(in_dim=in_dim)
train_from_tensors(
    joint_clean,
    bundle.X_train,
    joint_y_train,
    bundle.X_val,
    joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["joint"], "mlp_joint_clean"),
)

mt_clean = MultiTaskMLPModel(in_dim=in_dim)
train_from_tensors(
    mt_clean,
    bundle.X_train,
    mt_y_train,
    bundle.X_val,
    mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["multitask"], "mlp_multitask_clean"),
)

coord_clean = CoordinateMLPModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
)
train_from_tensors(
    coord_clean,
    bundle.X_train,
    coord_y_train,
    bundle.X_val,
    coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["coordinate"], "mlp_coordinate_clean"),
)

models_clean = {
    "mlp_joint": joint_clean,
    "mlp_multitask": mt_clean,
    "mlp_coordinate": coord_clean,
}

yval_by_model = {
    "mlp_joint": joint_y_val,
    "mlp_multitask": mt_y_val,
    "mlp_coordinate": coord_y_val,
}

eval_batch_sizes = {
    "mlp_joint": FINAL_TUNED_CFGS["mlp"]["joint"].get("val_batch_size", 512),
    "mlp_multitask": FINAL_TUNED_CFGS["mlp"]["multitask"].get("val_batch_size", 512),
    "mlp_coordinate": FINAL_TUNED_CFGS["mlp"]["coordinate"].get("val_batch_size", 512),
}

results_clean = stack_model_results(models_clean, yval_by_model, eval_batch_sizes, "clean_train")
results_clean


epoch=001 train_loss=0.3446 val_loss=0.4605 score=0.8677
epoch=005 train_loss=0.0271 val_loss=0.4676 score=0.8947
epoch=010 train_loss=0.0180 val_loss=0.5100 score=0.8938
epoch=015 train_loss=0.0067 val_loss=0.5283 score=0.9028
epoch=001 train_loss=0.3399 val_loss=0.4026 score=0.8821
epoch=005 train_loss=0.0300 val_loss=0.5386 score=0.8731
epoch=010 train_loss=0.0150 val_loss=0.5060 score=0.8992
epoch=015 train_loss=0.0092 val_loss=0.5785 score=0.9001
epoch=020 train_loss=0.0081 val_loss=0.6389 score=0.8974
epoch=025 train_loss=0.0077 val_loss=0.6324 score=0.8983
epoch=030 train_loss=0.0069 val_loss=0.6441 score=0.9019
epoch=035 train_loss=0.0071 val_loss=0.6235 score=0.9028
epoch=040 train_loss=0.0070 val_loss=0.6569 score=0.9010
epoch=001 train_loss=0.1244 val_loss=0.0297 score=-20.0418
epoch=005 train_loss=0.0274 val_loss=0.0174 score=-13.5294
epoch=010 train_loss=0.0186 val_loss=0.0167 score=-13.4287
epoch=015 train_loss=0.0163 val_loss=0.0138 score=-11.2813
epoch=020 train_loss=0.

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.907291,0.907291,0.997300,0.907291,0.466643,mlp_joint,clean_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.896490,0.896490,0.997300,0.896490,0.491018,mlp_joint,clean_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.863186,0.863186,0.992799,0.863186,0.624152,mlp_joint,clean_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.836184,0.836184,0.995500,0.836184,0.680629,mlp_joint,clean_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.834383,0.834383,0.993699,0.834383,0.676498,mlp_joint,clean_train,NaN,NaN,NaN,NaN
5,dropout_0.25,0.831683,0.831683,0.983798,0.834383,0.710301,mlp_joint,clean_train,NaN,NaN,NaN,NaN
6,dropout_0.30,0.796580,0.796580,0.981098,0.801080,0.839838,mlp_joint,clean_train,NaN,NaN,NaN,NaN
7,dropout_0.35,0.760576,0.760576,0.971197,0.766877,0.991439,mlp_joint,clean_train,NaN,NaN,NaN,NaN
8,dropout_0.40,0.708371,0.708371,0.951395,0.714671,1.164709,mlp_joint,clean_train,NaN,NaN,NaN,NaN
9,dropout_0.45,0.687669,0.687669,0.929793,0.703870,1.248821,mlp_joint,clean_train,NaN,NaN,NaN,NaN


## 2) Train on augmented/composite data


In [8]:
joint_aug = JointMLPModel(in_dim=in_dim)
train_from_tensors(
    joint_aug,
    X_train_composite,
    joint_y_composite,
    bundle.X_val,
    joint_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["joint"], "mlp_joint_aug"),
)

mt_aug = MultiTaskMLPModel(in_dim=in_dim)
train_from_tensors(
    mt_aug,
    X_train_composite,
    mt_y_composite,
    bundle.X_val,
    mt_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["multitask"], "mlp_multitask_aug"),
)

coord_aug = CoordinateMLPModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
)
train_from_tensors(
    coord_aug,
    X_train_composite,
    coord_y_composite,
    bundle.X_val,
    coord_y_val,
    device,
    make_cfg(FINAL_TUNED_CFGS["mlp"]["coordinate"], "mlp_coordinate_aug"),
)

models_aug = {
    "mlp_joint": joint_aug,
    "mlp_multitask": mt_aug,
    "mlp_coordinate": coord_aug,
}

results_aug = stack_model_results(models_aug, yval_by_model, eval_batch_sizes, "aug_train")
results_aug


epoch=001 train_loss=0.2667 val_loss=0.3344 score=0.9001
epoch=005 train_loss=0.0526 val_loss=0.6228 score=0.8830
epoch=010 train_loss=0.0193 val_loss=0.5530 score=0.9073
epoch=015 train_loss=0.0183 val_loss=0.6957 score=0.9028
epoch=020 train_loss=0.0102 val_loss=0.7253 score=0.9154
epoch=025 train_loss=0.0093 val_loss=0.7804 score=0.9145
epoch=030 train_loss=0.0067 val_loss=0.7117 score=0.9163
epoch=035 train_loss=0.0068 val_loss=0.7233 score=0.9172
epoch=040 train_loss=0.0058 val_loss=0.7904 score=0.9136
epoch=001 train_loss=0.2680 val_loss=0.3835 score=0.9010
epoch=005 train_loss=0.0570 val_loss=0.5601 score=0.9073
epoch=010 train_loss=0.0370 val_loss=0.4812 score=0.9073
epoch=015 train_loss=0.0201 val_loss=0.6329 score=0.9091
epoch=020 train_loss=0.0176 val_loss=0.6312 score=0.9100
epoch=025 train_loss=0.0113 val_loss=0.6997 score=0.9181
epoch=001 train_loss=0.0730 val_loss=0.0179 score=-13.8351
epoch=005 train_loss=0.0196 val_loss=0.0126 score=-11.4953
epoch=010 train_loss=0.0162

,scenario,score,joint_accuracy,building_accuracy,floor_accuracy,eval_loss,model,train_regime,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,clean,0.921692,0.921692,0.997300,0.921692,0.631600,mlp_joint,aug_train,NaN,NaN,NaN,NaN
1,dropout_0.05,0.916292,0.916292,0.997300,0.916292,0.635443,mlp_joint,aug_train,NaN,NaN,NaN,NaN
2,dropout_0.10,0.909991,0.909991,0.998200,0.909991,0.682436,mlp_joint,aug_train,NaN,NaN,NaN,NaN
3,dropout_0.15,0.904590,0.904590,0.998200,0.904590,0.733143,mlp_joint,aug_train,NaN,NaN,NaN,NaN
4,dropout_0.20,0.893789,0.893789,0.999100,0.893789,0.724766,mlp_joint,aug_train,NaN,NaN,NaN,NaN
5,dropout_0.25,0.881188,0.881188,0.994599,0.881188,0.748244,mlp_joint,aug_train,NaN,NaN,NaN,NaN
6,dropout_0.30,0.874888,0.874888,0.997300,0.875788,0.812759,mlp_joint,aug_train,NaN,NaN,NaN,NaN
7,dropout_0.35,0.855086,0.855086,0.995500,0.855986,0.973960,mlp_joint,aug_train,NaN,NaN,NaN,NaN
8,dropout_0.40,0.847885,0.847885,0.994599,0.849685,0.968203,mlp_joint,aug_train,NaN,NaN,NaN,NaN
9,dropout_0.45,0.821782,0.821782,0.987399,0.825383,1.050853,mlp_joint,aug_train,NaN,NaN,NaN,NaN


## 3) Compare `score` clean-train vs augmented-train


In [9]:
cmp = results_clean[["scenario", "model", "score"]].merge(
    results_aug[["scenario", "model", "score"]],
    on=["scenario", "model"],
    suffixes=("_clean_train", "_aug_train"),
)
cmp["delta_score"] = cmp["score_aug_train"] - cmp["score_clean_train"]
cmp


,scenario,model,score_clean_train,score_aug_train,delta_score
0,clean,mlp_joint,0.907291,0.921692,0.014401
1,dropout_0.05,mlp_joint,0.896490,0.916292,0.019802
2,dropout_0.10,mlp_joint,0.863186,0.909991,0.046805
3,dropout_0.15,mlp_joint,0.836184,0.904590,0.068407
4,dropout_0.20,mlp_joint,0.834383,0.893789,0.059406
5,dropout_0.25,mlp_joint,0.831683,0.881188,0.049505
6,dropout_0.30,mlp_joint,0.796580,0.874888,0.078308
7,dropout_0.35,mlp_joint,0.760576,0.855086,0.094509
8,dropout_0.40,mlp_joint,0.708371,0.847885,0.139514
9,dropout_0.45,mlp_joint,0.687669,0.821782,0.134113


In [10]:
cmp_pivot = cmp.pivot_table(
    index="scenario",
    columns="model",
    values="delta_score",
    aggfunc="first",
)
cmp_pivot


model,mlp_coordinate,mlp_joint,mlp_multitask
scenario,,,
bias5_only,1.036538,0.025203,0.026103
clean,0.970859,0.014401,0.014401
drop0.15_bias6_noise0,2.140708,0.049505,0.040504
drop0.25_bias3_noise1,3.396037,0.071107,0.072007
drop0.35_bias4_noise2,5.358545,0.110711,0.136814
drop0.40_bias5_noise3,6.910100,0.118812,0.131413
dropout_0.05,1.285831,0.019802,0.018902
dropout_0.10,2.014385,0.046805,0.039604
dropout_0.15,2.454671,0.068407,0.056706


In [11]:
# Optional: save the same artifacts used by the result notebooks/plot scripts.
OUT_DIR = Path("notebooks/logs/mlp_robustness") if (Path.cwd().name != "notebooks") else Path("logs/mlp_robustness")
OUT_DIR.mkdir(parents=True, exist_ok=True)

selected_cfg_df.to_csv(OUT_DIR / "tuned_hparams.csv", index=False)
results_clean.to_csv(OUT_DIR / "results_clean.csv", index=False)
results_aug.to_csv(OUT_DIR / "results_aug.csv", index=False)
cmp.to_csv(OUT_DIR / "cmp_delta_score.csv", index=False)
cmp_pivot.to_csv(OUT_DIR / "cmp_delta_score_pivot.csv")

print("Saved CSV outputs to:", OUT_DIR.resolve())


Saved CSV outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/mlp_robustness
